In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import pickle
import joblib

from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

In [2]:
from sklearn.metrics import precision_score, recall_score, confusion_matrix,  roc_curve, precision_recall_curve, accuracy_score, roc_auc_score,f1_score
from sklearn.metrics import roc_curve, auc

# 1. 导入数据

In [3]:
df_model_train = pickle.load(open('../results/df_model_train.pkl', 'rb'))
df_model_test = pickle.load(open('../results/df_model_test.pkl', 'rb'))

df_model_knn_train = pickle.load(open('../results/df_model_knn_train.pkl', 'rb'))
df_model_knn_test = pickle.load(open('../results/df_model_knn_test.pkl', 'rb'))

In [4]:
df_model_train

,Group01,Age,WBC,NEUT_No,LYM_No,MON_No,EOS_No,BASO_No,NRBC_No,RBC,...,MCH,MCHC,PLT,MPV,PDW,Plateletcrit_,CA125,NLR,PLR,SII
3910,0,34,5.60,3.35,1.64,0.38,0.20,0.03,0.008,4.32,...,29.6,346.0,259.0,7.7,16.5,20.0,NaN,2.042683,157.926829,529.054878
4568,0,35,5.82,3.90,1.54,0.28,0.05,0.05,0.005,3.80,...,30.3,338.0,189.0,8.7,16.2,16.4,NaN,2.532468,122.727273,478.636364
830,1,55,5.60,4.30,1.00,0.20,0.00,0.00,0.000,4.49,...,19.7,304.0,361.0,6.7,16.0,24.3,17.85,4.300000,361.000000,1552.300000
4294,0,35,6.90,3.60,2.70,0.50,0.09,0.02,0.010,4.92,...,27.2,321.0,211.0,8.6,16.0,18.1,NaN,1.333333,78.148148,281.333333
2787,0,24,7.70,4.80,2.30,0.50,0.10,0.00,0.010,4.34,...,31.2,334.0,366.0,7.7,15.7,28.2,NaN,2.086957,159.130435,763.826087
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4604,0,33,6.59,4.43,1.66,0.44,0.04,0.01,0.005,4.35,...,25.9,328.0,207.0,9.9,17.2,20.3,16.26,2.668675,124.698795,552.415663
1379,1,38,6.28,3.33,2.51,0.30,0.08,0.02,NaN,4.18,...,31.1,342.0,333.0,9.5,9.5,32.0,12.30,1.326693,132.669323,441.788845
3855,0,33,6.10,3.20,2.30,0.30,0.30,0.00,0.010,5.08,...,25.9,320.0,242.0,8.8,18.0,21.2,NaN,1.391304,105.217391,336.695652
2102,1,35,4.13,2.14,1.52,0.30,0.06,0.10,NaN,4.60,...,17.4,282.0,135.0,10.5,14.8,14.0,NaN,1.407895,88.815789,190.065789


# 2. 测序保存的模型结果是否一致

In [7]:
rf_clf = joblib.load('../results/randomforest/test/rf_clf.pkl')
rf_clf

RandomForestClassifier(random_state=2025)

In [8]:
X_test = df_model_knn_test.drop('Group01', axis=1)
y_test = df_model_knn_test['Group01']
# 预测
y_pred = rf_clf.predict(X_test)
y_proba = rf_clf.predict_proba(X_test)[:, 1]

In [9]:
metrics_dict = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "AUC": roc_auc_score(y_test, y_proba),
    "Model": "RF"
}
print(metrics_dict)

{'Accuracy': 0.8892508143322475, 'Precision': 0.9041769041769042, 'Recall': 0.8538283062645011, 'F1': 0.8782816229116945, 'AUC': 0.9444528623514372, 'Model': 'RF'}


# 3. 德隆检验

In [10]:
def compute_midrank(x):
    x = np.asarray(x)
    order = np.argsort(x)
    sorted_x = x[order]
    n = len(x)
    ranks = np.zeros(n, dtype=float)

    i = 0
    while i < n:
        j = i
        while j < n and sorted_x[j] == sorted_x[i]:
            j += 1
        ranks[i:j] = 0.5 * (i + j - 1) + 1
        i = j

    out = np.empty(n, dtype=float)
    out[order] = ranks
    return out


def fast_delong(predictions_sorted_transposed, label_1_count):
    m = label_1_count
    n = predictions_sorted_transposed.shape[1] - m

    pos_preds = predictions_sorted_transposed[:, :m]
    neg_preds = predictions_sorted_transposed[:, m:]

    k = predictions_sorted_transposed.shape[0]

    tx = np.zeros((k, m))
    ty = np.zeros((k, n))
    tz = np.zeros((k, m + n))

    for r in range(k):
        tx[r] = compute_midrank(pos_preds[r])
        ty[r] = compute_midrank(neg_preds[r])
        tz[r] = compute_midrank(predictions_sorted_transposed[r])

    aucs = (tz[:, :m].sum(axis=1) / m - (m + 1) / 2) / n

    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m

    sx = np.atleast_2d(np.cov(v01))
    sy = np.atleast_2d(np.cov(v10))

    delong_cov = sx / m + sy / n
    return aucs, delong_cov


def delong_test(y_true, pred1, pred2):
    """
    DeLong test for two correlated ROC AUCs.

    Parameters
    ----------
    y_true : array-like, shape (n_samples,)
        Binary labels, 0/1.
    pred1 : array-like, shape (n_samples,)
        Predicted probability or score from model 1.
    pred2 : array-like, shape (n_samples,)
        Predicted probability or score from model 2.

    Returns
    -------
    result : dict
        auc1, auc2, auc_diff, z, p_value
    """

    y_true = np.asarray(y_true).astype(int)
    pred1 = np.asarray(pred1, dtype=float)
    pred2 = np.asarray(pred2, dtype=float)

    assert set(np.unique(y_true)).issubset({0, 1})
    assert len(y_true) == len(pred1) == len(pred2)

    order = np.argsort(-y_true)
    y_sorted = y_true[order]

    preds_sorted = np.vstack([
        pred1[order],
        pred2[order]
    ])

    n_pos = int(np.sum(y_sorted))
    n_neg = len(y_sorted) - n_pos

    if n_pos == 0 or n_neg == 0:
        raise ValueError("y_true must contain both positive and negative samples.")

    aucs, cov = fast_delong(preds_sorted, n_pos)

    auc1, auc2 = aucs
    auc_diff = auc1 - auc2

    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]

    if var <= 0:
        raise ValueError(f"Non-positive variance in DeLong test: {var}")

    z = auc_diff / np.sqrt(var)
    p_value = 2 * stats.norm.sf(abs(z))

    return {
        "auc1": float(auc1),
        "auc2": float(auc2),
        "auc_diff": float(auc_diff),
        "z": float(z),
        "p_value": float(p_value),
    }

In [11]:
methods = [('XGBoost with NaN', '../results/xgboost/test/xgb_clf_model_withnan.pkl', df_model_test, df_model_test), 
           ('LightGBM', '../results/lgbm/test/lgbm_clf.pkl', df_model_knn_test, df_model_knn_test),
           ('XGBoost', '../results/xgboost/test/xgb_clf_model.pkl', df_model_knn_test, df_model_knn_test),
           ('RandomForest', '../results/randomforest/test/rf_clf.pkl', df_model_knn_test, df_model_knn_test),
           ('LogisticRegression', '../results/lr/test/lr_clf.pkl', df_model_knn_test, df_model_knn_test),
           ('MLP', '../results/mlp/test/mlp.pkl', df_model_knn_test, df_model_knn_test),
           ('SVM', '../results/svm/test/svm_clf.pkl', df_model_knn_test, df_model_knn_test),
           ('KNN', '../results/knn/test/knn_clf.pkl', df_model_knn_test, df_model_knn_test)]
for method1, model_file1, X_test1, y_test1 in methods:
    X_test = X_test1.drop('Group01', axis=1)
    y_test = y_test1['Group01']
    clf = joblib.load(model_file1)
    y_pro1 = clf.predict_proba(X_test)[:, 1]
    for method2, model_file2, X_test2, y_test2 in methods:
        if method1 == method2:
            continue
        print(f"### '{method1}' v.s. '{method2}'")
        X_test = X_test2.drop('Group01', axis=1)
        y_test = y_test2['Group01']
        clf = joblib.load(model_file2)
        y_pro2 = clf.predict_proba(X_test)[:, 1]
        res = delong_test(y_test, y_pro1, y_pro2)
        print(f"AUC1={res['auc1']:.4f}, AUC2={res['auc2']:.4f}, Z={res['z']:.4f}, P-value={res['p_value']:.6f}")

### 'XGBoost with NaN' v.s. 'LightGBM'
AUC1=0.9902, AUC2=0.9669, Z=5.6916, P-value=0.000000
### 'XGBoost with NaN' v.s. 'XGBoost'
AUC1=0.9902, AUC2=0.9624, Z=5.6261, P-value=0.000000
### 'XGBoost with NaN' v.s. 'RandomForest'
AUC1=0.9902, AUC2=0.9445, Z=6.6393, P-value=0.000000
### 'XGBoost with NaN' v.s. 'LogisticRegression'
AUC1=0.9902, AUC2=0.8660, Z=10.6166, P-value=0.000000
### 'XGBoost with NaN' v.s. 'MLP'
AUC1=0.9902, AUC2=0.8358, Z=11.8866, P-value=0.000000
### 'XGBoost with NaN' v.s. 'SVM'
AUC1=0.9902, AUC2=0.7333, Z=15.8512, P-value=0.000000
### 'XGBoost with NaN' v.s. 'KNN'
AUC1=0.9902, AUC2=0.7014, Z=17.1397, P-value=0.000000
### 'LightGBM' v.s. 'XGBoost with NaN'
AUC1=0.9669, AUC2=0.9902, Z=-5.6916, P-value=0.000000
### 'LightGBM' v.s. 'XGBoost'
AUC1=0.9669, AUC2=0.9624, Z=1.9507, P-value=0.051092
### 'LightGBM' v.s. 'RandomForest'
AUC1=0.9669, AUC2=0.9445, Z=5.1149, P-value=0.000000
### 'LightGBM' v.s. 'LogisticRegression'
AUC1=0.9669, AUC2=0.8660, Z=9.4497, P-value=0.000